# 03 – Parameter estimation (PINN + MCMC)

Rolling-window estimation of R0, Delta (mean delay) and the epidemic onset tau0 with the pretrained PINN. At every week the parameters are estimated on the observations up to that week, via MCMC. One notebook for the five cases of the paper, chosen with `CASE`:

| `CASE` | Data | Observable | Delta | Look-back (weeks) | Weights |
|---|---|---|---|---|---|
| `"synthetic"` (R0 = 1.5 / 2.0, noise `"light"` / `"mild"`) | `data/synthetic/` | new infections | fixed to 21 | all the history | yes |
| `"real"` | `data/epi/london/deaths.csv` | new deaths | estimated | 5 | no |
| `"real"`, `CASE_OVERRIDES = dict(dt_train=None)` | same | new deaths | estimated | all the history (25) | yes |

The harmonic weights of the Poisson likelihood are on by default when the look-back is the whole history, off otherwise (`weight` in `CASE_OVERRIDES` changes it). The name of the results folder tells the look-back, e.g. `real_London_lookback5`, `real_London_lookback25`, `synthetic_R0_2.0_noise_light`.

Each case has these defaults in `pinnvsmech.inference.cases.PRESETS`; `CASE_OVERRIDES` changes them.

For the synthetic scenarios the rolling windows are the weeks 4-30 (R0 = 1.5) and 4-20 (R0 = 2.0); in addition one window is estimated at week 34 (R0 = 1.5) / 24 (R0 = 2.0), used for the density of R0 at the last week (notebook 07). The chain of every week is saved, then the posterior samples of the weeks are collected in `mcmc_distributions.pkl` (with the derived Rt and I at the first week of the window). The estimation is repeated for every seed in `SEEDS`: 100 independent runs, or only one if `SEEDS = [10]`. A run takes a few minutes: with `RESUME = True` the seeds already completed are skipped, so an interrupted run can be restarted.

Results: `outputs/posteriors/pinn/<case>/seed_<seed>/`. To use them as the results of the paper, copy them to `precomputed/posteriors/pinn/`.

No plots here.

In [1]:
import json
import os

import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

from pinnvsmech.constants import N_age
from pinnvsmech.inference.cases import load_case
from pinnvsmech.inference.mcmc import run_rolling_mcmc_inference, save_mcmc_chain_tails, export_last_week_samples
from pinnvsmech.pinn.NN_free_params import NN
from pinnvsmech.paths import PRECOMPUTED_DIR, OUTPUTS_DIR

torch.set_default_dtype(torch.float64)

## Settings

In [4]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 1.5                     # synthetic only: 1.5 or 2.0
NOISE = "light"                    # synthetic only: "light" or "mild"
CASE_OVERRIDES = {}                # e.g. dict(dt_train=8, weight=True, delta_fixed=None); see PRESETS

SEEDS = [10]                       # one run (seed 10)
# SEEDS = list(range(100))         # 100 independent runs (seeds 0..99)
RESUME = True                      # skip the seeds already completed

NSIM = 10000                       # MCMC iterations for every week
PRIORS = "uniform"                 # "uniform" or "normal"
LOSS = "poisson"                   # "poisson", "wmape" or "mse"
SAMPLE_STRATEGY = "tail"           # posterior samples: last NSAMPLE iterations of the chain
NSAMPLE = 1000
SAVE_LAST_WEEK_CSV = False         # also save the samples of the last week as CSV

PINN_WEIGHTS = PRECOMPUTED_DIR / "pretrained_pinn" / "pretrained_pinn.pth"
# After a new training (notebook 02): OUTPUTS_DIR / "pinn" / "models" / "<RUN_NAME>.pth"
CHAINS_DIR = OUTPUTS_DIR / "posteriors" / "pinn"

## Data and PINN

In [5]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
print(case.label)
print(case.settings())

synthetic_R0_1.5_noise_light
{'kind': 'synthetic', 'label': 'synthetic_R0_1.5_noise_light', 'observable': 'infections', 'delta_fixed': 21, 'dt_train': 35, 'weight': True, 't0_range': [3, 30], 't0_last': 33, 't0_estimation': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 33], 'n_weeks': 35}


In [6]:
model = NN(activation=nn.Tanh, dimensions=[3, 100, 100, 100, 100, 100, 6 * len(N_age)])
model.load_state_dict(torch.load(PINN_WEIGHTS, weights_only=True))
model.eval()

NN(
  (stack): Sequential(
    (0): Linear(in_features=3, out_features=100, bias=True)
    (1): Tanh()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): Tanh()
    (4): Linear(in_features=100, out_features=100, bias=True)
    (5): Tanh()
    (6): Linear(in_features=100, out_features=100, bias=True)
    (7): Tanh()
    (8): Linear(in_features=100, out_features=100, bias=True)
    (9): Tanh()
    (10): Linear(in_features=100, out_features=96, bias=True)
    (11): Sigmoid()
  )
)

## Estimation

In [7]:
case_dir = CHAINS_DIR / case.label
os.makedirs(case_dir, exist_ok=True)

# Settings of the run, saved with the results
with open(case_dir / "run_config.json", "w") as f:
    json.dump({**case.settings(), "nsim": NSIM, "priors": PRIORS, "loss": LOSS,
               "sample_strategy": SAMPLE_STRATEGY, "nsample": NSAMPLE,
               "seeds": list(SEEDS), "pinn_weights": PINN_WEIGHTS.name}, f, indent=2)

for seed in tqdm(SEEDS, desc=case.label):
    seed_dir = case_dir / f"seed_{seed:03d}"
    if RESUME and (seed_dir / "mcmc_distributions.pkl").exists():
        continue

    # One chain for every week
    run_rolling_mcmc_inference(
        N_age=N_age, dt_train=case.dt_train, output_dir=seed_dir,
        dates=case.dates, obs=case.obs, pinn_model=model, t0_range=case.t0_estimation(),
        delta_fixed=case.delta_fixed, observable=case.observable,
        weight=case.weight, nsim=NSIM, priors=PRIORS, loss=LOSS, seed=seed,
    )

    # Posterior samples of all the weeks, with Rt and I
    save_mcmc_chain_tails(
        path=seed_dir, N_age=N_age, dt_train=case.dt_train, dates=case.dates,
        t0_range=case.t0_estimation(), pinn_model=model, delta_fixed=case.delta_fixed,
        sample_strategy=SAMPLE_STRATEGY, nsample=NSAMPLE, nsim=NSIM,
    )

    if SAVE_LAST_WEEK_CSV:
        export_last_week_samples(seed_dir)

synthetic_R0_1.5_noise_light:   0%|          | 0/1 [00:00<?, ?it/s]

File saved in: /home/mrama/SINNs/PINNs_epi/Repo_GitHub/PINNvsMechanistic-models-for-epidemic-modeling-and-forecasting/outputs/posteriors/pinn/synthetic_R0_1.5_noise_light/seed_010/mcmc_distributions.pkl
